# UMUD EXP72 heavy thin-structure segmentation overnight run

This is the deliberately heavier follow-up to EXP59. It is not just more epochs on the same binary-mask task.

What is different:
- trains the thin-line target with soft/dilated targets instead of only razor-thin binary masks,
- sweeps validation thresholds instead of assuming 0.5,
- optionally decodes thin masks with skeleton-style postprocessing,
- saves logs/status/debug masks so an interrupted Kaggle session leaves evidence.

Run setup:
1. Add the competition input: **UMUD Challenge: Muscle Architecture in Ultrasound Data**.
2. Set **Accelerator = GPU** and **Internet = On**.
3. If Kaggle offers persistence, persist files. Variables are optional.
4. Run All.

Expected runtime: this is intentionally heavy. On a T4, one run can take hours; all four can approach an overnight session. Change `MAX_RUNS` if you only want the first serious candidate.


In [ ]:
# Fast data preflight. This fails before package install/training if the competition input is not attached.
from pathlib import Path
import os

IMG_EXTS = ('.tif', '.tiff', '.png', '.jpg', '.jpeg', '.bmp')
LEAVES = {
    'apo_img': ['apo_images_new_model_v1'],
    'apo_msk': ['apo_masks_new_model_v1'],
    'fasc_img': ['fasc_images_new_model_v1'],
    'fasc_msk': ['fasc_masks_new_model_v1'],
    'test': ['test_set_v2', 'test_images_v2'],
}
TERMINAL_LEAVES = {'apo_images_new_model_v1', 'apo_masks_new_model_v1', 'fasc_images_new_model_v1', 'fasc_masks_new_model_v1', 'test_set_v2'}

def list_images(d):
    try:
        return [p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
    except OSError:
        return []

def print_tree(root, max_depth=3, depth=0, budget=None):
    budget = [300] if budget is None else budget
    try:
        entries = sorted(root.iterdir())
    except OSError:
        return
    for p in entries:
        if budget[0] <= 0:
            return
        budget[0] -= 1
        print('  ' * depth + ('[D] ' if p.is_dir() else '    ') + p.name)
        if p.is_dir() and depth < max_depth:
            print_tree(p, max_depth, depth + 1, budget)

def index_root(root):
    index = {}
    for dirpath, dirnames, _files in os.walk(root):
        for dn in dirnames:
            index.setdefault(dn, []).append(Path(dirpath) / dn)
        dirnames[:] = [d for d in dirnames if d not in TERMINAL_LEAVES]
    return index

def resolve_key(index, key):
    for leaf in LEAVES[key]:
        cands = index.get(leaf, [])
        if key == 'test':
            cands = sorted(cands, key=lambda d: len(list_images(d)), reverse=True)
            cands = [d for d in cands if list_images(d)]
        if cands:
            return cands[0]
    return None

roots = sorted(p for p in Path('/kaggle/input').iterdir() if p.is_dir()) if Path('/kaggle/input').exists() else []
print('Mounted roots:')
for root in roots:
    print(' -', root)

resolved = None
best_n = -1
for root in roots:
    idx = index_root(root)
    dirs = {key: resolve_key(idx, key) for key in LEAVES}
    n = sum(v is not None for v in dirs.values())
    if n > best_n:
        resolved = dirs
        best_n = n
resolved = resolved or {key: None for key in LEAVES}

print('\nResolved data folders:')
for key, path in resolved.items():
    count = len(list_images(path)) if path else 0
    print(f'  {key:8s} -> {path} ({count} images)' if path else f'  {key:8s} -> None')
missing = [key for key, path in resolved.items() if path is None]
if missing:
    print('\nMounted tree for debugging:')
    for root in roots or [Path('/kaggle/input')]:
        print(f'[root] {root}')
        print_tree(root)
    raise RuntimeError(f'Missing UMUD folders: {missing}. Attach the competition input, then Run All again.')

assert len(list_images(resolved['test'])) == 309, f"expected 309 test images, found {len(list_images(resolved['test']))}"
print('\nPreflight OK: competition data is mounted and test set has 309 images.')


In [ ]:
# Install dependencies and download current repo scripts. Internet must be On.
import subprocess
import sys
import urllib.request
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch', 'albumentations'])

RAW_BASE = 'https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture'
SCRIPTS = [
    'segment_then_measure.py',
    'tick_calibration.py',
    'scale_ticks.py',
    'subpixel_spacing.py',
    'scale_ocr.py',
]
for script in SCRIPTS:
    urllib.request.urlretrieve(f'{RAW_BASE}/{script}', script)
    assert Path(script).exists(), f'{script} failed to download; check Internet = On'

EXPECTED_VERSION = '2026-06-13.03'
got = 'MISSING'
for line in Path('segment_then_measure.py').read_text(encoding='utf-8').splitlines():
    if line.startswith('PIPELINE_VERSION'):
        got = line.split('"')[1]
        break
print(f'>>> downloaded pipeline VERSION: {got}   (expected {EXPECTED_VERSION}) <<<')
assert got == EXPECTED_VERSION, 'STALE script: GitHub raw served an old file. Wait 1-2 minutes and re-run this cell.'

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')


In [ ]:
# Run the heavy thin-structure matrix.
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

WORK = Path('/kaggle/working')
LOG_DIR = WORK / 'seg72_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
STATUS_PATH = WORK / 'seg72_thin_structure_status.json'
SUMMARY_PATH = WORK / 'seg72_thin_structure_summary.csv'

COMMON_ENV = {
    'PYTHONUNBUFFERED': '1',
    'UMUD_SCALE_ROUTER': '1',
    'UMUD_TTA': '1',
    'UMUD_USE_CALIBRATED_MT': '1',
    'UMUD_USE_CALIBRATED_FL': '0',
    'UMUD_FRAGMENT_FL': '1',
    'UMUD_FL_RECENTER': '1',
    'UMUD_FL_IDENTITY_BLEND': '0',
    'UMUD_TOP_BOUNDARY_MODE': 'robust_triangle',
    'UMUD_MT_MODE': 'perp_center',
    'UMUD_AUTO_THRESHOLD': '1',
    'UMUD_THRESHOLD_SWEEP': '0.12,0.16,0.20,0.24,0.30,0.36,0.44,0.52',
    'UMUD_APO_TARGET_MODE': 'binary',
    'UMUD_APO_MASK_THRESHOLD': '0.5',
    'UMUD_FASC_MIN_ANG': '4',
    'UMUD_SAVE_PRED_DEBUG': '24',
}

RUNS = [
    {
        'run_id': 'seg72_01_soft5_tversky_640_unetpp',
        'reason': 'primary thin-structure run: soft Gaussian target, Tversky-style loss, threshold sweep, skeleton-dilate decoding',
        'env': {
            'UMUD_EPOCHS': '45',
            'UMUD_IMG_SIZE': '640',
            'UMUD_TTA_EXTRA_SIZE': '768',
            'UMUD_BATCH_SIZE': '3',
            'UMUD_MODEL_ARCH': 'unetplusplus',
            'UMUD_MODEL_ENCODER': 'resnet34',
            'UMUD_LOSS_MODE': 'bce_tversky',
            'UMUD_AUG_LEVEL': 'heavy',
            'UMUD_FASC_TARGET_MODE': 'soft5',
            'UMUD_FASC_POSTPROCESS': 'skeleton_dilate',
            'UMUD_FASC_MIN_AREA': '14',
            'UMUD_WEIGHTS_TAG': 'seg72_01',
        },
    },
    {
        'run_id': 'seg72_02_dilate_soft5_tversky_640_unetpp',
        'reason': 'same architecture but thicker soft target: tests whether near-miss gradients beat pure soft centerline',
        'env': {
            'UMUD_EPOCHS': '45',
            'UMUD_IMG_SIZE': '640',
            'UMUD_TTA_EXTRA_SIZE': '768',
            'UMUD_BATCH_SIZE': '3',
            'UMUD_MODEL_ARCH': 'unetplusplus',
            'UMUD_MODEL_ENCODER': 'resnet34',
            'UMUD_LOSS_MODE': 'dice_tversky',
            'UMUD_AUG_LEVEL': 'heavy',
            'UMUD_FASC_TARGET_MODE': 'dilate_soft5',
            'UMUD_FASC_POSTPROCESS': 'skeleton_dilate',
            'UMUD_FASC_MIN_AREA': '14',
            'UMUD_WEIGHTS_TAG': 'seg72_02',
        },
    },
    {
        'run_id': 'seg72_03_soft7_focal_768_unet',
        'reason': 'higher effective resolution and wider soft target; slower but checks whether 512/640 resize is losing thin detail',
        'env': {
            'UMUD_EPOCHS': '36',
            'UMUD_IMG_SIZE': '768',
            'UMUD_TTA_EXTRA_SIZE': '896',
            'UMUD_BATCH_SIZE': '2',
            'UMUD_MODEL_ARCH': 'unet',
            'UMUD_MODEL_ENCODER': 'resnet34',
            'UMUD_LOSS_MODE': 'dice_focal',
            'UMUD_AUG_LEVEL': 'strong',
            'UMUD_FASC_TARGET_MODE': 'soft7',
            'UMUD_FASC_POSTPROCESS': 'skeleton_dilate',
            'UMUD_FASC_MIN_AREA': '12',
            'UMUD_WEIGHTS_TAG': 'seg72_03',
        },
    },
    {
        'run_id': 'seg72_04_recall_dilate3_tversky_640_unetpp',
        'reason': 'recall-biased variant: lower area gate plus dilated binary target, meant to reveal missed fragments rather than polish masks',
        'env': {
            'UMUD_EPOCHS': '45',
            'UMUD_IMG_SIZE': '640',
            'UMUD_TTA_EXTRA_SIZE': '768',
            'UMUD_BATCH_SIZE': '3',
            'UMUD_MODEL_ARCH': 'unetplusplus',
            'UMUD_MODEL_ENCODER': 'resnet34',
            'UMUD_LOSS_MODE': 'dice_tversky',
            'UMUD_AUG_LEVEL': 'heavy',
            'UMUD_FASC_TARGET_MODE': 'dilate3',
            'UMUD_FASC_POSTPROCESS': 'skeleton',
            'UMUD_FASC_POS_WEIGHT': '2.5',
            'UMUD_FASC_MIN_AREA': '10',
            'UMUD_WEIGHTS_TAG': 'seg72_04',
        },
    },
]

MAX_RUNS = 4
MAX_WALL_HOURS = 11.5
FORCE_RERUN = False


def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')


def load_summaries():
    if SUMMARY_PATH.exists():
        try:
            return pd.read_csv(SUMMARY_PATH).to_dict('records')
        except Exception:
            return []
    return []


def write_status(payload):
    payload = dict(payload)
    payload['updated_at_utc'] = now_iso()
    STATUS_PATH.write_text(json.dumps(payload, indent=2, default=str), encoding='utf-8')


def append_or_replace_summary(rows, row):
    rows = [r for r in rows if r.get('run_id') != row.get('run_id')]
    rows.append(row)
    pd.DataFrame(rows).to_csv(SUMMARY_PATH, index=False)
    return rows


def valid_submission(path):
    if not path.exists():
        return False
    try:
        sub = pd.read_csv(path)
    except Exception:
        return False
    return sub.shape == (309, 4) and sub['image_id'].is_unique and not sub[['pa_deg', 'fl_mm', 'mt_mm']].isna().any().any()


def collect_outputs(run_id):
    produced = []
    for src_name, dst_name in [
        ('submission_segmentation.csv', f'submission_{run_id}.csv'),
        ('calibration_measurement_debug.csv', f'calibration_measurement_debug_{run_id}.csv'),
    ]:
        src = WORK / src_name
        if src.exists():
            dst = WORK / dst_name
            shutil.copy2(src, dst)
            produced.append(dst.name)
    return produced


def summarize_outputs(run_id, elapsed_min=None, status='ok', error=''):
    stats = {}
    sub_path = WORK / f'submission_{run_id}.csv'
    dbg_path = WORK / f'calibration_measurement_debug_{run_id}.csv'
    if sub_path.exists():
        sub = pd.read_csv(sub_path)
        stats.update({
            'rows': len(sub),
            'pa_mean': float(sub['pa_deg'].mean()),
            'pa_std': float(sub['pa_deg'].std()),
            'fl_mean': float(sub['fl_mm'].mean()),
            'fl_std': float(sub['fl_mm'].std()),
            'mt_mean': float(sub['mt_mm'].mean()),
            'mt_std': float(sub['mt_mm'].std()),
        })
    if dbg_path.exists():
        dbg = pd.read_csv(dbg_path)
        if 'calibration_method' in dbg:
            stats['scaled_images'] = int((dbg['calibration_method'] != 'none').sum())
            stats['calibration_methods'] = json.dumps(dbg['calibration_method'].value_counts(dropna=False).head(12).to_dict())
        for col in ['fl_fragment_n', 'n_fascicles']:
            if col in dbg:
                stats[f'{col}_mean'] = float(pd.to_numeric(dbg[col], errors='coerce').mean())
    return {'run_id': run_id, 'status': status, 'elapsed_min': elapsed_min, 'error': error, **stats}


summaries = load_summaries()
matrix_start = time.time()
write_status({'state': 'matrix_start', 'max_runs': MAX_RUNS, 'max_wall_hours': MAX_WALL_HOURS, 'force_rerun': FORCE_RERUN})

for run in RUNS[:MAX_RUNS]:
    if (time.time() - matrix_start) / 3600.0 > MAX_WALL_HOURS:
        msg = f'stopping before {run["run_id"]}: wall budget reached'
        print(msg)
        write_status({'state': 'wall_budget_stop', 'message': msg, 'summary_path': str(SUMMARY_PATH)})
        break

    run_id = run['run_id']
    log_path = LOG_DIR / f'{run_id}.log'
    final_sub = WORK / f'submission_{run_id}.csv'
    if valid_submission(final_sub) and not FORCE_RERUN:
        print('\nSKIP completed', run_id, '->', final_sub.name)
        summaries = append_or_replace_summary(summaries, summarize_outputs(run_id, status='skipped_completed'))
        continue

    print('\n' + '=' * 96)
    print('START', run_id)
    print(run['reason'])
    print('log:', log_path)
    print('=' * 96)

    env = os.environ.copy()
    env.update(COMMON_ENV)
    env.update(run['env'])
    config = {'run_id': run_id, 'reason': run['reason'], 'env': {k: env[k] for k in sorted(set(COMMON_ENV) | set(run['env']))}}
    (WORK / f'run_config_{run_id}.json').write_text(json.dumps(config, indent=2), encoding='utf-8')

    start = time.time()
    write_status({'state': 'running', 'run_id': run_id, 'started_at_utc': now_iso(), 'log_path': str(log_path), 'config': config})
    summaries = append_or_replace_summary(summaries, {'run_id': run_id, 'status': 'running', 'elapsed_min': 0.0, 'error': '', 'log_path': str(log_path)})

    with log_path.open('w', encoding='utf-8', errors='replace') as log:
        log.write(f'START {run_id} {now_iso()}\n')
        log.write(json.dumps(config, indent=2) + '\n')
        log.flush()
        proc = subprocess.Popen([sys.executable, '-u', 'segment_then_measure.py'], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        returncode = proc.wait()
        log.write(f'END {run_id} {now_iso()} returncode={returncode}\n')

    elapsed_min = (time.time() - start) / 60.0
    produced = collect_outputs(run_id)
    if returncode == 0 and valid_submission(final_sub):
        row = summarize_outputs(run_id, elapsed_min=elapsed_min, status='ok', error='')
        row['produced'] = json.dumps(produced)
        print('FINISHED', run_id, f'in {elapsed_min:.1f} min')
    else:
        row = summarize_outputs(run_id, elapsed_min=elapsed_min, status='failed', error=f'returncode={returncode}; produced={produced}')
        row['produced'] = json.dumps(produced)
        print('FAILED', run_id, row['error'])

    row['log_path'] = str(log_path)
    summaries = append_or_replace_summary(summaries, row)
    write_status({'state': row['status'], 'run_id': run_id, 'elapsed_min': elapsed_min, 'log_path': str(log_path), 'summary_path': str(SUMMARY_PATH)})
    print('Summary so far:')
    print(pd.DataFrame(summaries).to_string(index=False))

write_status({'state': 'matrix_complete', 'summary_path': str(SUMMARY_PATH), 'runs': summaries})
print('\nHEAVY THIN-STRUCTURE MATRIX COMPLETE OR RECORDED.')
print('Summary file:', SUMMARY_PATH)
print('Status file:', STATUS_PATH)
print('Logs:', LOG_DIR)


In [ ]:
# Bundle submissions, debug files, configs, summaries, logs/status, debug masks, and weights into one zip.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
import os

WORK = Path('/kaggle/working')
zip_path = WORK / 'umud_seg72_thin_structure_outputs.zip'
patterns = [
    'submission_*.csv',
    'calibration_measurement_debug_*.csv',
    'seg72_thin_structure_summary.csv',
    'seg72_thin_structure_status.json',
    'run_config_*.json',
    'seg_*.pt',
    'seg72_logs/*.log',
    'pred_debug_*/*.png',
]
include_paths = []
for pattern in patterns:
    include_paths.extend(sorted(WORK.glob(pattern)))
include_paths = sorted({p.resolve(): p for p in include_paths if p.exists()}.values(), key=lambda p: str(p))

with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as zf:
    for path in include_paths:
        zf.write(path, arcname=str(path.relative_to(WORK)))

print('Created:', zip_path.name, f'({zip_path.stat().st_size:,} bytes)')
print('Contents:')
for path in include_paths:
    print(' -', path.relative_to(WORK), f'({path.stat().st_size:,} bytes)')

os.chdir(WORK)
display(FileLink(zip_path.name))
for path in include_paths:
    rel = path.relative_to(WORK)
    if path.name.startswith('submission_') or path.name == 'seg72_thin_structure_summary.csv' or path.name == 'seg72_thin_structure_status.json':
        display(FileLink(str(rel)))
